# Choke Points in LLM Reasoning - Results Walkthrough

Narrated tour of the saved results. Run top to bottom; every cell loads
outputs from `results/` (nothing is recomputed here) and explains them in
prose with figures inline. For the examiner demo: problem -> architecture
-> Figure 4 -> Figure 7 -> this notebook -> paper PDF.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

TABLES = Path('../results/tables')
FIGURES = Path('../results/figures')
print('Tables:', sorted(p.name for p in TABLES.glob('*')) if TABLES.exists() else 'NONE YET - run the pipeline first')
print('Figures:', sorted(p.name for p in FIGURES.glob('*.png')) if FIGURES.exists() else 'NONE YET')

## 1. Where failures originate (E1)

Descriptive statistics per difficulty tier plus the pooled row. The
`mean_steps_remaining` column bounds the mitigation opportunity: steps
during which intervention is still possible.

In [ ]:
desc = pd.read_csv(TABLES / 'table1_descriptive.csv')
desc

## 2. Headline figure: relative onset (F4)

If errors concentrated rather than spreading uniformly, a detection window
exists. The KS test in the next cell makes that claim defensible.

In [ ]:
display(Image(str(FIGURES / 'fig4_relative_error_position_HEADLINE.png')))
print('Read it as: histogram shows WHERE errors begin; CDF shows how much of the chain must be watched to catch X% of errors.')

## 3. Statistical tests (E2)

KS against Uniform(0,1) tests concentration; Mann-Whitney checks the
length confound; Kruskal-Wallis checks difficulty dependence. Effect sizes
sit beside every p-value.

In [ ]:
import json
stats = json.loads((TABLES / 'table_stats_tests.json').read_text())
print(json.dumps(stats, indent=2))
ks = stats['ks_uniform']
print(f"\nIn words: KS D={ks['statistic_D']:.4f}, p={ks['p_value']:.3e} -", "errors concentrate." if ks['p_value'] < 0.05 else "uniformity not rejected; the window is weaker than hypothesised.")

## 4. Core result: predictability curve (F7)

AUROC for eventual failure as a function of observed prefix, surface vs
lexical here (hidden-state joins the overlay once the GPU stage runs).
Read the 25% point for the abstract's headline claim.

In [ ]:
curve = pd.read_csv(TABLES / 'table2_prefix_auroc.csv')
curve
display(Image(str(FIGURES / 'fig7_auroc_vs_prefix_KEY.png')))

## 5. Timing and trust (E5/E6): lead time + calibration

Delta = e - k* tells whether alarms precede errors; ECE tells whether
risk numbers can drive a threshold policy.

In [ ]:
lead = pd.read_csv(TABLES / 'table_lead_summary.csv')
lead
display(Image(str(FIGURES / 'fig9_lead_time.png')))
display(Image(str(FIGURES / 'fig10_reliability.png')))

## 6. Cost of acting (E8): recovery vs corruption

A policy that raises recovery while raising corruption equally is
worthless - both are reported at matched token budgets, with the oracle as
upper bound.

In [ ]:
pol = pd.read_csv(TABLES / 'table_intervention.csv')
pol
display(Image(str(FIGURES / 'fig11_recovery_vs_corruption.png')))
display(Image(str(FIGURES / 'fig12_cost_frontier.png')))